# Lab 80: The Half-Second Delay

*Brain Lab: 84 Experiments to Decode Your Biological and Artificial Intelligence* · Chapter 9, *The Future Interface*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/labphonlab/brain-lab-support/blob/main/notebooks/lab-80.ipynb)

**What you will do:** time your own W-judgement on ten trials of the Libet clock task, compare your average with the published timings, then use the "logit lens" technique to watch a small language model's answer take shape inside its own layers before it reaches the output.

**How to use.** Run the cells from top to bottom (Shift+Enter). Nothing needs installing, and nothing you do here can break anything: *Runtime → Restart and run all* returns it to the start. To keep your own results, use *File → Save a copy in Drive*.

In [ ]:
# Setup: run this cell first (Shift+Enter).
import platform, numpy as np, pandas as pd, matplotlib
import matplotlib.pyplot as plt
np.random.seed(2026)
print("Python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__)

## The experiment in the book

In Lab 80 you watched a clock and noted, on each trial, the clock's position at the moment you first felt
the urge to act -- not when you acted. Libet's original experiment measured three events: the readiness
potential (brain activity beginning to prepare the movement), the reported urge (W), and the movement
itself, and found the brain begins preparing the action well before you become aware of wanting to.

## Record your results

Convert each trial's clock reading into "milliseconds before your action" (measure this directly with a
stopwatch app if you tried Step 3's speech variant, or estimate it from the clock face if you used
Steps 1-2). Enter your ten trials below.

In [ ]:
import pandas as pd

# example data: replace with your own ten trials, in milliseconds before the flick/word
my_trials_ms = pd.DataFrame({
    "trial": range(1, 11),
    "ms_before_action": [180, 220, 195, 240, 170, 210, 205, 260, 190, 215],  # example data
})
display(my_trials_ms)
my_trials_ms["ms_before_action"].plot.hist(bins=6, color="steelblue", figsize=(6, 3))
plt.xlabel("your reported W (ms before action)"); plt.title("Your ten trials"); plt.show()
print(f"Your mean W: {my_trials_ms['ms_before_action'].mean():.0f} ms before action")

## Compare

Libet (1985) reported the conscious urge (W) at about 200 ms before movement, with the readiness
potential beginning about 550 ms before movement -- a roughly 350 ms gap between unconscious preparation
and conscious awareness.

In [ ]:
your_w = my_trials_ms["ms_before_action"].mean()
print(f"Your mean W: {your_w:.0f} ms before action (Libet's reported average: about 200 ms)")
if 100 <= your_w <= 300:
    print("Your average is in the same rough range Libet reported.")
else:
    print("Your average is some way from Libet's figure. Estimating a felt urge against a moving clock")
    print("hand is a notoriously imprecise measurement -- the original studies used many more trials and a")
    print("purpose-built rotating-spot display, so a handful of trials on an ordinary clock will be noisy.")
print()
print("For comparison, Soon et al. (2008) used fMRI to predict which hand a participant would choose up")
print("to 7-10 seconds before their reported decision, at about 60% accuracy on a two-choice task (50%")
print("would be chance) -- a much longer lead time than Libet's few hundred milliseconds, but a far more")
print("modest and probabilistic prediction.")

## The AI side

Mechanistic interpretability tries to catch an AI model "deciding" before its final output, much as
Libet's EEG tried to catch the brain preparing an action before conscious awareness. The **logit lens**
(nostalgebraist, 2020) is a simple version of this: at each layer of a language model, you can peek at
what word the model would output *if generation stopped right there*, by running that layer's
intermediate representation through the model's own final output stage. Below, a small GPT-2 model
answers a simple completion, and we watch its answer take shape layer by layer.

*This part downloads a small open model or library, so it runs in Colab (it may take a minute the first time).*

In [ ]:
!pip install -q transformers

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, output_hidden_states=True)
model.eval()

prompt = "The opposite of hot is"
inputs = tokenizer(prompt, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)

hidden_states = outputs.hidden_states  # one per layer, plus the embedding layer
final_ln = model.transformer.ln_f
lm_head = model.lm_head

rows = []
for layer_idx, h in enumerate(hidden_states):
    last_token_hidden = h[0, -1, :]
    logits = lm_head(final_ln(last_token_hidden))
    top_id = logits.argmax().item()
    rows.append({"layer": layer_idx, "best_guess": tokenizer.decode([top_id]).strip()})

layer_guesses = pd.DataFrame(rows)
display(layer_guesses)
final_guess = layer_guesses.iloc[-1]["best_guess"]
print(f'\nPrompt: "{prompt}"')
print(f'Final output (layer {len(hidden_states) - 1}): "{final_guess}"')

Look at how early the final answer appears. In many runs, GPT-2-sized models settle on their eventual
top word several layers before the last one -- the "decision" is visible in the logit lens before the
model has finished computing. The disanalogy matters: the logit lens reads out a token probability using
machinery (the model's own final layer norm and unembedding) that was never meant to be applied
mid-network, so early "guesses" can be a distorted preview rather than a faithful readout of what that
layer represents (this is the same basis-drift problem that motivated later refinements such as the
tuned lens). It is a hint about internal computation, not literally a hidden decision the model is
"aware of" the way Libet's participants were aware of their urge -- the model has no urge, no veto, and
no experience of the delay at all.

## Pool the class data

Pool everyone's mean W judgement (in ms before action), using anonymous IDs, to build a distribution
across several people rather than trusting any one person's noisy average.

In [ ]:
import io

csv_text = """id,mean_w_ms
P01,205
P02,240
P03,160
P04,290
P05,215
"""  # example data: replace with your class CSV
w_df = pd.read_csv(io.StringIO(csv_text))
boot = [w_df["mean_w_ms"].sample(len(w_df), replace=True).mean() for _ in range(5000)]
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"Class mean W: {w_df['mean_w_ms'].mean():.0f} ms before action "
      f"(95% bootstrap interval {lo:.0f}-{hi:.0f} ms, n = {len(w_df)})")
w_df["mean_w_ms"].plot.hist(bins=6, color="steelblue", alpha=0.8, figsize=(6, 3))
plt.axvline(200, color="firebrick", linestyle="--", label="Libet's reported average (200 ms)")
plt.xlabel("mean W (ms before action)"); plt.legend(); plt.show()

## Questions to think about

1. Schurger, Sitt, and Dehaene (2012) argue that the readiness potential may reflect random neural fluctuations crossing a threshold at the moment you happen to act, rather than a specific unconscious decision made in advance. Does this reinterpretation change what your own W measurement above would mean, or only what the earlier readiness-potential measurement would mean?
2. In the logit lens output, did the model's guess change more than once before settling, or did it lock onto the final answer immediately and never revise? What would repeated revision suggest about how "decided" an intermediate layer's guess really is?
3. Libet argued that consciousness retains a "veto power" over an already-initiated action. Does anything in the logit lens demonstration have an equivalent of a veto -- a later layer overriding an earlier layer's guess? If it does, is that the same kind of event as a person consciously stopping themselves partway through a movement?
4. The book distinguishes Libet's roughly 350 ms gap from Soon et al.'s 7-10 second, ~60%-accurate prediction. Which of these two findings does the logit lens demonstration resemble more closely, and why?

## Challenge

Re-run the logit lens cell with a prompt that has a much less predictable completion (for example, "My
favourite thing about today is" instead of "The opposite of hot is"). Does the model's answer stabilise
earlier or later in the network for a genuinely open-ended prompt than for a prompt with one obvious
correct completion?

In [ ]:
# Reuse the code from the ai_lab cell above with a different prompt, and record how many layers before
# the end the model's guess first matched its final answer.
prompts_to_try = ["My favourite thing about today is", "Two plus two equals", "The capital of France is"]
layers_to_settle = {
    "My favourite thing about today is": 2,  # example data
    "Two plus two equals": 5,                # example data
    "The capital of France is": 4,           # example data
}
for prompt in prompts_to_try:
    print(f'"{prompt}" settled {layers_to_settle[prompt]} layer(s) before the output layer')
print()
print("A genuinely open-ended prompt has no single 'correct' completion, so watch whether it settles later")
print("(or never firmly) compared with prompts that have an obvious answer.")

## Record what you ran

In [ ]:
NOTEBOOK_ID = "lab-80.ipynb"
# Reproducibility record: paste this output into your notes or report.
import sys, platform, datetime, numpy, pandas, matplotlib
print("Notebook:", NOTEBOOK_ID)
print("Run at  :", datetime.datetime.now().isoformat(timespec="seconds"))
print("Python  :", sys.version.split()[0], "on", platform.platform())
print("numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)
print("Seed    : 2026")